In [1]:
import os, sqlite3

In [ ]:
DB_PATH = "experiment.db"

def fresh(pk_decl):
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
    conn = sqlite3.connect(DB_PATH, autocommit=False)
    conn.execute(f"CREATE TABLE clients (id {pk_decl}, name TEXT NOT NULL)")
    conn.commit()
    return conn

def rows(conn):
    return conn.execute("SELECT id, name FROM clients ORDER BY id").fetchall()

In [3]:
conn = fresh("INTEGER PRIMARY KEY AUTOINCREMENT")
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm B",))
conn.commit()

observer = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)

def look(label):
    print(f"{label:<28} writer sees: {rows(conn)}")
    print(f"{'':<28} observer sees: {rows(observer)}")
    observer.rollback() # end the observer's read transaction so it never blocks the writer

look("start")

start                        writer sees: [(1, 'Farm A'), (2, 'Farm B')]
                             observer sees: [(1, 'Farm A'), (2, 'Farm B')]


In [4]:
conn.execute("UPDATE clients SET name = ? WHERE id = ?", ("Farm A (renamed)", 1))
look("after UPDATE")

conn.execute("DELETE FROM clients WHERE id = ?", (2, ))
look("after DELETE")

after UPDATE                 writer sees: [(1, 'Farm A (renamed)'), (2, 'Farm B')]
                             observer sees: [(1, 'Farm A'), (2, 'Farm B')]
after DELETE                 writer sees: [(1, 'Farm A (renamed)')]
                             observer sees: [(1, 'Farm A'), (2, 'Farm B')]


In [5]:
conn.rollback()
look("after rollback")

after rollback               writer sees: [(1, 'Farm A'), (2, 'Farm B')]
                             observer sees: [(1, 'Farm A'), (2, 'Farm B')]


In [6]:
conn.execute("UPDATE clients SET name = ? WHERE id = ?", ("Farm A (renamed)", 1))
conn.commit()
look("after UPDATE + commit")

conn.execute("DELETE FROM clients WHERE id = ?", (2,))
conn.rollback()
look("after DELETE + rollback")

after UPDATE + commit        writer sees: [(1, 'Farm A (renamed)'), (2, 'Farm B')]
                             observer sees: [(1, 'Farm A (renamed)'), (2, 'Farm B')]
after DELETE + rollback      writer sees: [(1, 'Farm A (renamed)'), (2, 'Farm B')]
                             observer sees: [(1, 'Farm A (renamed)'), (2, 'Farm B')]


In [7]:
conn.close()
observer.close()